# 02_train_baseline_model (NACC) — Updated

Trains a simple baseline 4-class classifier using `output/cohort.csv` produced by `01_build_cohort`.


In [22]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


OUT_DIR  = os.path.join("..", "output")
DATA_PATH = os.path.join(OUT_DIR, "cohort.csv")

df = pd.read_csv(DATA_PATH)

print("Loaded:", df.shape)
print(df["label"].value_counts(dropna=False))
df.head()


Loaded: (37007, 55)
label
Censored         22339
NoDementia_5y    10816
AD                2948
MixedOrOther       445
PD_LBD             283
Vascular           176
Name: count, dtype: int64


,NACCID,baseline_date,NACCAGE,SEX,RACE,HISPANIC,EDUC,DIABETES,HYPERTEN,HYPERCHO,...,BPSYS_missing,BPDIAS_missing,STROKCOG_missing,CVDCOG_missing,STKIMAG_missing,CVDIMAG_missing,CSFABETA,CSFPTAU,CSFTTAU,abs_days_from_baseline
0,NACC000011,2006-04-17,62,2,1,0,16.0,0,0,0,...,0,0,1,1,1,1,NaN,NaN,NaN,NaN
1,NACC000034,2015-07-16,79,2,1,0,15.0,0,1,0,...,0,0,1,1,0,1,NaN,NaN,NaN,NaN
2,NACC000067,2012-05-23,60,1,1,0,18.0,0,0,0,...,0,0,1,0,1,1,NaN,NaN,NaN,NaN
3,NACC000073,2016-05-20,44,2,1,0,18.0,0,0,0,...,1,0,1,1,0,1,NaN,NaN,NaN,NaN
4,NACC000144,2010-08-17,79,1,1,0,18.0,0,1,1,...,0,0,1,0,1,1,NaN,NaN,NaN,NaN


In [23]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42

## Keep only the 4-class task

In [24]:
df4 = df.copy()  # if your df is already the cohort dataframe; otherwise remove this line

# Keep only non-censored outcome classes for training
KEEP = {"AD", "NoDementia_5y", "PD_LBD", "Vascular"}
df4 = df4[df4["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Drop IDs/labels
y = df4["label"].astype(str)
drop_cols = [c for c in ["NACCID", "baseline_date", "label"] if c in df4.columns]
X_all = df4.drop(columns=drop_cols)

# Treat these as categorical codes (even though stored as ints)
CAT_CODE_VARS = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD"] if c in X_all.columns]

# --- Clean special missing codes BEFORE converting to categories ---
STD_MISS = {-4, 9}

for c in CAT_CODE_VARS:
    s = pd.to_numeric(X_all[c], errors="coerce")
    s = s.mask(s.isin(list(STD_MISS)), np.nan)
    X_all[c] = s

# RACE often uses 99 for "Missing/Unknown" in some dictionary versions
if "RACE" in X_all.columns:
    X_all["RACE"] = X_all["RACE"].mask(X_all["RACE"].isin([99]), np.nan)

# Convert categorical code columns to sklearn-friendly OBJECT strings
# Keep missing as np.nan (not "nan" string).
for c in CAT_CODE_VARS:
    s = pd.to_numeric(X_all[c], errors="coerce")
    # keep NaN as NaN
    X_all[c] = s.map(lambda v: np.nan if pd.isna(v) else str(int(v))).astype(object)

# Education: keep numeric for modeling, but create education bin for subgroup analysis
if "EDUC" in X_all.columns:
    educ_num = pd.to_numeric(X_all["EDUC"], errors="coerce")
    X_all["EDUC_BIN"] = np.where(educ_num <= 12, "<=12", ">12").astype(object)

# --- Final sklearn compatibility cleanup: convert any pd.NA to np.nan ---
X_all = X_all.replace({pd.NA: np.nan})

# Optional: booleans can confuse pipelines; convert to 0/1
for c in X_all.columns:
    if X_all[c].dtype == "bool":
        X_all[c] = X_all[c].astype(int)

print("Cat columns (object):", [c for c in X_all.columns if X_all[c].dtype == "object"][:20])


4-class rows: 14223
label
NoDementia_5y    10816
AD                2948
PD_LBD             283
Vascular           176
Name: count, dtype: int64
Cat columns (object): ['SEX', 'RACE', 'HISPANIC', 'EDUC_BIN']


In [25]:
print("Dtypes of categorical code vars:")
for c in CAT_CODE_VARS:
    print(c, X_all[c].dtype, X_all[c].dropna().unique()[:5])


Dtypes of categorical code vars:
SEX object ['2' '1']
RACE object ['1' '5' '2' '3' '50']
HISPANIC object ['0' '1']


In [26]:
print("NACCUDSD in X_all?", "NACCUDSD" in X_all.columns)


NACCUDSD in X_all? False


## Split + preprocess + train

Train/test split

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

labels = ["AD", "NoDementia_5y", "PD_LBD", "Vascular"]


Preprocessing pipelines (sparse for LR/boosting)

In [28]:
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("string")]
num_cols = [c for c in X_train.columns if c not in cat_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

print("Num cols:", len(num_cols), "Cat cols:", len(cat_cols))


Num cols: 49 Cat cols: 4


Model A: Logistic Regression (final settings)

In [29]:
from sklearn.linear_model import LogisticRegression

custom_w = {
    "NoDementia_5y": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_train, y_train)
pred_lr = clf_lr.predict(X_test)

print("=== Logistic Regression (custom weights, C=0.1) ===")
print(classification_report(y_test, pred_lr, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr, labels=labels))


=== Logistic Regression (custom weights, C=0.1) ===
               precision    recall  f1-score   support

           AD     0.7175    0.7492    0.7330       590
NoDementia_5y     0.9484    0.9427    0.9455      2163
       PD_LBD     0.2609    0.2105    0.2330        57
     Vascular     0.1515    0.1429    0.1471        35

     accuracy                         0.8780      2845
    macro avg     0.5196    0.5113    0.5146      2845
 weighted avg     0.8769    0.8780    0.8773      2845

Balanced accuracy: 0.5113020538840931
[[ 442   99   30   19]
 [ 115 2039    2    7]
 [  37    6   12    2]
 [  22    6    2    5]]


Model B: Boosted trees + SHAP (XGBoost preferred, LightGBM fallback)

Try importing XGBoost / LightGBM

In [ ]:
from sklearn.preprocessing import LabelEncoder

labels = ["AD", "NoDementia_5y", "PD_LBD", "Vascular"]

le = LabelEncoder()
le.fit(labels)

y_train_enc = le.transform(y_train)   # 0..3
y_test_enc  = le.transform(y_test)

print("Label mapping:", {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))})

Label mapping: {np.str_('AD'): np.int64(0), np.str_('NoDementia_5y'): np.int64(1), np.str_('PD_LBD'): np.int64(2), np.str_('Vascular'): np.int64(3)}


In [31]:
use_model = None

try:
    from xgboost import XGBClassifier
    use_model = "xgb"
    print("Using XGBoost.")
except Exception as e:
    try:
        from lightgbm import LGBMClassifier
        use_model = "lgbm"
        print("Using LightGBM.")
    except Exception as e2:
        use_model = None
        print("Neither xgboost nor lightgbm found. Install one:")
        print("  pip install xgboost shap")
        print("or")
        print("  pip install lightgbm shap")


Using XGBoost.


Train boosted model (with sample weights)

In [32]:
if use_model is not None:
    # sample weights based on your custom weights (strings)
    w_train = y_train.map(custom_w).astype(float).values

    if use_model == "xgb":
        booster = XGBClassifier(
            n_estimators=600,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective="multi:softprob",
            num_class=len(labels),
            eval_metric="mlogloss",
            random_state=RANDOM_STATE
        )
    else:
        booster = LGBMClassifier(
            n_estimators=1200,
            max_depth=-1,
            num_leaves=31,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=0.1,
            objective="multiclass",
            random_state=RANDOM_STATE
        )

    clf_boost = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", booster)
    ])

    # IMPORTANT: fit with encoded labels
    clf_boost.fit(X_train, y_train_enc, model__sample_weight=w_train)

    # predict encoded and decode to string labels
    pred_boost_enc = clf_boost.predict(X_test)
    pred_boost = le.inverse_transform(pred_boost_enc.astype(int))

    print("=== Boosted Trees (custom sample_weight) ===")
    print(classification_report(y_test, pred_boost, digits=4))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_boost))
    print(confusion_matrix(y_test, pred_boost, labels=labels))

=== Boosted Trees (custom sample_weight) ===
               precision    recall  f1-score   support

           AD     0.7469    0.8102    0.7772       590
NoDementia_5y     0.9532    0.9501    0.9516      2163
       PD_LBD     0.4359    0.2982    0.3542        57
     Vascular     0.3000    0.0857    0.1333        35

     accuracy                         0.8974      2845
    macro avg     0.6090    0.5360    0.5541      2845
 weighted avg     0.8920    0.8974    0.8934      2845

Balanced accuracy: 0.5360496848505995
[[ 478   87   19    6]
 [ 105 2055    3    0]
 [  33    6   17    1]
 [  24    8    0    3]]


In [33]:
print("Boost model classes_:", getattr(clf_boost.named_steps["model"], "classes_", None))

Boost model classes_: [0 1 2 3]


Interpretability Part 1: Logistic Regression coefficient “top drivers”

In [34]:
def get_feature_names(preprocessor: ColumnTransformer):
    # numeric names
    num_names = preprocessor.named_transformers_["num"].get_feature_names_out(num_cols)
    # categorical names
    ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    cat_names = ohe.get_feature_names_out(cat_cols)
    return np.concatenate([num_names, cat_names])

# Fit already done; extract coefficients
pre = clf_lr.named_steps["preprocess"]
model = clf_lr.named_steps["model"]
feat_names = get_feature_names(pre)

coef = pd.DataFrame(model.coef_, columns=feat_names, index=model.classes_)
coef_path = os.path.join(OUT_DIR, "lr_coefficients_all_classes.csv")
coef.to_csv(coef_path)
print("Saved:", coef_path)

# Save top drivers per class
for cls in model.classes_:
    s = coef.loc[cls].sort_values(ascending=False)
    top_pos = s.head(15).reset_index()
    top_pos.columns = ["feature", "coef"]
    top_neg = s.tail(15).sort_values().reset_index()
    top_neg.columns = ["feature", "coef"]

    top_pos.to_csv(os.path.join(OUT_DIR, f"lr_top_positive_{cls}.csv"), index=False)
    top_neg.to_csv(os.path.join(OUT_DIR, f"lr_top_negative_{cls}.csv"), index=False)

print("Saved per-class top coefficient CSVs in output/.")


Saved: ..\output\lr_coefficients_all_classes.csv
Saved per-class top coefficient CSVs in output/.


Interpretability Part 2: SHAP for boosted trees (global + local)

In [37]:
if use_model is not None:
    import shap
    import numpy as np
    import matplotlib.pyplot as plt

    # Transform X once (same features model sees)
    X_test_mat  = clf_boost.named_steps["preprocess"].transform(X_test)
    X_train_mat = clf_boost.named_steps["preprocess"].transform(X_train)

    feat_names_boost = get_feature_names(clf_boost.named_steps["preprocess"])

    # ---- SHAP (robust) ----
    # TreeExplainer fails for your XGBoost multiclass base_score vector, so use model-agnostic Explainer.
    # For speed, we use a small background sample from TRAIN.
    bg_n = min(300, X_train_mat.shape[0])
    rng = np.random.default_rng(RANDOM_STATE)
    bg_idx = rng.choice(X_train_mat.shape[0], size=bg_n, replace=False)
    background = X_train_mat[bg_idx]

    # For speed, also limit the explained test points (you can increase later)
    explain_n = min(600, X_test_mat.shape[0])
    X_explain = X_test_mat[:explain_n]

    # shap.Explainer expects a callable model; use predict_proba on the *pipeline*
    # but it expects input in original feature space (X_test), not preprocessed.
    # So we wrap a function that takes the preprocessed matrix and calls the model directly.
    base_model = clf_boost.named_steps["model"]

    # Need predict_proba on the underlying model using already-transformed matrices:
    def model_proba_on_matrix(M):
        return base_model.predict_proba(M)

    explainer = shap.Explainer(model_proba_on_matrix, background, feature_names=feat_names_boost)
    sv = explainer(X_explain)  # sv.values: (n_samples, n_features, n_classes)

    # --- Global summary plots for each class ---
    for i, cls in enumerate(labels):
        plt.figure()
        shap.summary_plot(
            sv.values[:, :, i],
            X_explain,
            feature_names=feat_names_boost,
            show=False,
            max_display=20
        )
        p = os.path.join(FIG_DIR, f"shap_summary_{cls}.png")
        plt.tight_layout()
        plt.savefig(p, dpi=200)
        plt.close()
        print("Saved:", p)

    # --- Local explanations: one example per predicted class ---
    # Get predicted classes for the explained subset
    pred_cls_enc = base_model.predict(X_explain)
    pred_cls = le.inverse_transform(pred_cls_enc.astype(int))

    for target in labels:
        idxs = np.where(pred_cls == target)[0]
        if len(idxs) == 0:
            continue
        j = int(idxs[0])

        # Waterfall for class index i
        i = labels.index(target)

        plt.figure()
        shap.plots.waterfall(
            shap.Explanation(
                values=sv.values[j, :, i],
                base_values=sv.base_values[j, i] if sv.base_values.ndim == 2 else sv.base_values[i],
                data=X_explain[j],
                feature_names=feat_names_boost
            ),
            max_display=15,
            show=False
        )
        p = os.path.join(FIG_DIR, f"shap_waterfall_pred_{target}.png")
        plt.tight_layout()
        plt.savefig(p, dpi=200)
        plt.close()
        print("Saved:", p)


PermutationExplainer explainer: 601it [02:16,  4.21it/s]                         
C:\Users\branjan\AppData\Local\Temp\ipykernel_15864\2746796708.py:39: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved: ..\output\figures\shap_summary_AD.png


C:\Users\branjan\AppData\Local\Temp\ipykernel_15864\2746796708.py:39: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved: ..\output\figures\shap_summary_NoDementia_5y.png


C:\Users\branjan\AppData\Local\Temp\ipykernel_15864\2746796708.py:39: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved: ..\output\figures\shap_summary_PD_LBD.png


C:\Users\branjan\AppData\Local\Temp\ipykernel_15864\2746796708.py:39: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved: ..\output\figures\shap_summary_Vascular.png
Saved: ..\output\figures\shap_waterfall_pred_AD.png
Saved: ..\output\figures\shap_waterfall_pred_NoDementia_5y.png
Saved: ..\output\figures\shap_waterfall_pred_PD_LBD.png
Saved: ..\output\figures\shap_waterfall_pred_Vascular.png


Subgroup performance (Sex, Race, Education bins)

In [39]:
def subgroup_report(model_pipe, X_test, y_test, group_series, group_name, le=None):
    rows = []
    for g in sorted(group_series.dropna().unique()):
        mask = (group_series == g)
        if mask.sum() < 30:
            continue

        yt = y_test[mask]
        yp = model_pipe.predict(X_test[mask])

        # If model outputs encoded numeric classes, decode to strings
        if le is not None:
            # yp could be numpy array of ints, or floats like 0.0, 1.0
            try:
                if np.issubdtype(np.array(yp).dtype, np.number):
                    yp = le.inverse_transform(np.array(yp).astype(int))
            except Exception:
                pass

        rows.append({
            "group_name": group_name,
            "group_value": str(g),
            "n": int(mask.sum()),
            "balanced_acc": balanced_accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro"),
            "recall_AD": ( (yp == "AD") & (yt == "AD") ).sum() / max(1, (yt == "AD").sum()),
            "recall_NoDementia": ( (yp == "NoDementia_5y") & (yt == "NoDementia_5y") ).sum() / max(1, (yt == "NoDementia_5y").sum()),
            "recall_PD_LBD": ( (yp == "PD_LBD") & (yt == "PD_LBD") ).sum() / max(1, (yt == "PD_LBD").sum()),
            "recall_Vascular": ( (yp == "Vascular") & (yt == "Vascular") ).sum() / max(1, (yt == "Vascular").sum()),
        })
    return pd.DataFrame(rows)


# Create subgroup variables from *raw X_test columns* (not encoded)
subgroups = {}

if "SEX" in X_test.columns:
    subgroups["SEX"] = X_test["SEX"]
if "RACE" in X_test.columns:
    # Optional: create White vs Non-White if you want
    subgroups["RACE"] = X_test["RACE"]
if "EDUC_BIN" in X_test.columns:
    subgroups["EDUC_BIN"] = X_test["EDUC_BIN"]

# Logistic regression subgroup table
tables = []
for name, s in subgroups.items():
    tables.append(subgroup_report(clf_lr, X_test, y_test, s, name))
sub_lr = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
p_lr = os.path.join(OUT_DIR, "subgroup_metrics_logreg.csv")
sub_lr.to_csv(p_lr, index=False)
print("Saved:", p_lr)
display(sub_lr.head(20))

# Boosted model subgroup table (if trained)
if use_model is not None:
    tables = []
    for name, s in subgroups.items():
        tables.append(subgroup_report(clf_boost, X_test, y_test, s, name, le=le))
    sub_boost = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
    p_boost = os.path.join(OUT_DIR, "subgroup_metrics_boosted.csv")
    sub_boost.to_csv(p_boost, index=False)
    print("Saved:", p_boost)
    display(sub_boost.head(20))


Saved: ..\output\subgroup_metrics_logreg.csv


,group_name,group_value,n,balanced_acc,macro_f1,recall_AD,recall_NoDementia,recall_PD_LBD,recall_Vascular
0,SEX,1,1160,0.536289,0.546272,0.728522,0.922693,0.266667,0.227273
1,SEX,2,1685,0.430919,0.428595,0.769231,0.954445,0.000000,0.000000
2,RACE,1,2309,0.501572,0.505686,0.742857,0.941210,0.222222,0.100000
3,RACE,2,406,0.590816,0.539791,0.736111,0.960486,0.000000,0.666667
4,RACE,5,84,0.440365,0.426013,0.842105,0.919355,0.000000,0.000000
5,EDUC_BIN,<=12,464,0.475143,0.475281,0.784173,0.916399,0.200000,0.000000
6,EDUC_BIN,>12,2381,0.514875,0.521320,0.738359,0.947084,0.212766,0.161290


Saved: ..\output\subgroup_metrics_boosted.csv


,group_name,group_value,n,balanced_acc,macro_f1,recall_AD,recall_NoDementia,recall_PD_LBD,recall_Vascular
0,SEX,1,1160,0.538630,0.553497,0.776632,0.931421,0.355556,0.090909
1,SEX,2,1685,0.491031,0.505876,0.842809,0.961058,0.083333,0.076923
2,RACE,1,2309,0.541246,0.562571,0.802041,0.948127,0.314815,0.100000
3,RACE,2,406,0.446502,0.445090,0.819444,0.966565,0.000000,0.000000
4,RACE,5,84,0.449491,0.424587,0.894737,0.903226,0.000000,0.000000
5,EDUC_BIN,<=12,464,0.584457,0.602894,0.848921,0.938907,0.300000,0.250000
6,EDUC_BIN,>12,2381,0.528140,0.543506,0.798226,0.951944,0.297872,0.064516


In [42]:
def subgroup_class_counts(y, group):
    t = pd.crosstab(group, y)
    return t

for name, s in subgroups.items():
    print("\n==", name, "==")
    display(subgroup_class_counts(y_test, s))



== SEX ==


label,AD,NoDementia_5y,PD_LBD,Vascular
SEX,,,,
1,291,802,45,22
2,299,1361,12,13



== RACE ==


label,AD,NoDementia_5y,PD_LBD,Vascular
RACE,,,,
1,490,1735,54,30
2,72,329,2,3
3,0,13,0,0
4,0,3,0,0
5,19,62,1,2
50,8,14,0,0



== EDUC_BIN ==


label,AD,NoDementia_5y,PD_LBD,Vascular
EDUC_BIN,,,,
<=12,139,311,10,4
>12,451,1852,47,31


Ablation study (prove the model improves as you add real clinical signal)

In [41]:
# Define feature groups (adjust if needed based on columns you actually have)
DEMOG = [c for c in ["NACCAGE","SEX","RACE","HISPANIC","EDUC","NACCUDSD"] if c in X_all.columns]
MED = [c for c in [
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","APNEA","DEP2YRS",
    "BPSYS","BPDIAS","CVCHF","CVANGINA","CVHATT","PREVSTK","STROKCOG","CVDCOG","STKIMAG",
    "CVDIMAG","CVDIMAG1","CVDIMAG2","CVDIMAG3","CVDIMAG4"
] if c in X_all.columns]
COG_FUNC = [c for c in ["NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON","CDRSUM","CDRGLOB"] if c in X_all.columns]

ABLATIONS = {
    "Demographics": DEMOG,
    "Demog+Medical": DEMOG + MED,
    "Demog+Medical+Cognition": DEMOG + MED + COG_FUNC,
    "Full": list(X_all.columns)
}

def train_eval_lr_on_cols(cols):
    Xtr = X_train[cols].copy()
    Xte = X_test[cols].copy()

    # Ensure categorical code vars are OBJECT (not pandas "string") and missing is np.nan
    cat_codes = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD", "EDUC_BIN"] if c in cols and c in Xtr.columns]
    for c in cat_codes:
        # Convert values to object strings, preserve missing as np.nan
        Xtr[c] = Xtr[c].map(lambda v: np.nan if pd.isna(v) else str(v)).astype(object)
        Xte[c] = Xte[c].map(lambda v: np.nan if pd.isna(v) else str(v)).astype(object)

    # Also convert any pd.NA left in the frame to np.nan
    Xtr = Xtr.replace({pd.NA: np.nan})
    Xte = Xte.replace({pd.NA: np.nan})

    cat_cols_ = [c for c in Xtr.columns if Xtr[c].dtype == "object"]
    num_cols_ = [c for c in Xtr.columns if c not in cat_cols_]

    pre_ = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), num_cols_),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_cols_),
        ],
        remainder="drop"
    )

    clf = Pipeline(steps=[
        ("preprocess", pre_),
        ("model", LogisticRegression(max_iter=8000, class_weight=custom_w, C=0.1))
    ])

    clf.fit(Xtr, y_train)
    yp = clf.predict(Xte)

    return {
        "set": "",
        "n_features": len(cols),
        "macro_f1": f1_score(y_test, yp, average="macro"),
        "balanced_acc": balanced_accuracy_score(y_test, yp)
    }


rows = []
for name, cols in ABLATIONS.items():
    if len(cols) == 0:
        continue
    r = train_eval_lr_on_cols(cols)
    r["set"] = name
    rows.append(r)

ab = pd.DataFrame(rows).sort_values("n_features")
p_ab = os.path.join(OUT_DIR, "ablation_logreg.csv")
ab.to_csv(p_ab, index=False)
print("Saved:", p_ab)
display(ab)


Saved: ..\output\ablation_logreg.csv


,set,n_features,macro_f1,balanced_acc
0,Demographics,5,0.288424,0.289557
1,Demog+Medical,27,0.364269,0.357739
2,Demog+Medical+Cognition,35,0.477782,0.484929
3,Full,53,0.514646,0.511302
